### RAG : OCI ADK Agent + Knowledge base OCI Object Storage Buckets

In [ ]:
# Confirm object storage access
import oci
config = oci.config.from_file("~/.oci/config", "DEFAULT")
object_storage = oci.object_storage.ObjectStorageClient(config)
namespace = object_storage.get_namespace().data
print("You can access your object storage bucket in namespace "namespace)

In [ ]:
import oci
import json
from oci.addons.adk import Agent, AgentClient, tool
from oci.object_storage import ObjectStorageClient

In [ ]:
CONFIG_PROFILE = "DEFAULT"
BUCKET_NAME = "demo-agent-kb"
OBJECT_NAME = "policies.json"     # employees policies data file loaded in oci bucket

In [ ]:
config = oci.config.from_file("~/.oci/config", CONFIG_PROFILE)
object_storage = ObjectStorageClient(config)
namespace = object_storage.get_namespace().data

In [ ]:
# Tool creation to fetch data from bucket

@tool
def retrieve_policy(policy_name: str) -> dict:
    """Retrieve policy content by name."""

    # Connect to bucket
    response = object_storage.get_object(namespace, BUCKET_NAME, OBJECT_NAME)
    
    # read the file in bucket
    retrived_data = json.loads(response.data.content.decode("utf-8"))
    return {"content": retrived_data.get(policy_name, "Policy not found.")}
    

In [ ]:
client = AgentClient(auth_type="api_key", profile=CONFIG_PROFILE, region="us-chicago-1")
agent = Agent(
    client=client,
        agent_endpoint_id="ocid1.genaiagentendpoint.oc1.us-chicago-1.amaaaaaa2fm4ibaapplmaoebbhl2bkcp6ezpskriykxqmiwwxfhajbqjcayq",
    instructions="You are a company policy assistant. Retrieve policies as requested.",
    tools=[retrieve_policy]
)
agent.setup()

response = agent.run("Show me the leave policy.")
print(response.data["message"]["content"]["text"])
